# Bitcoin Price Predictor — live training in SnowflakeSecond-by-second BTCUSDT tape replayed as a **live market feed**, a **CPU MarketAnalyser** turning it into signals, and a **GPU Price Predictor** forecasting theprice **30 minutes ahead** — retraining every 30 minutes of simulated market time.**This notebook needs no internet.** The dataset ships with the repository ascompact `.npz` chunks (or is pulled once from an internal stage); every libraryused is either in the Snowflake Anaconda channel or in the container runtimeimage. Nothing is downloaded at run time.| stage | runs on | what it does ||---|---|---|| Market Simulator | CPU | replays the 1-second tape as if it were streaming now || Market Analyser  | CPU | 40 causal signals: trend, momentum, volatility, order flow, clock || Price Predictor  | GPU | causal TCN → GRU → (μ, σ, direction), predicts t+1800 s || Stability filter | CPU | confidence-weighted EMA + slew limiter, so the forecast is tradeable |The model size, the number of DDP ranks sharing the GPU and the batch width are**derived from the hardware this notebook lands on** — nothing to tune.

## 1 · BootstrapPoint Python at `src/`. Works whether the repo was cloned, staged, or attached as a Snowflake git repository.

In [ ]:
import os, sys, glob, time, json
import numpy as np

# --- find the repository root (clone / stage mount / git repository) -------
CANDIDATES = [
    ".", "..",
    "/home/app/pridictior_snow4",
    "/tmp/pridictior_snow4",
]
# Snowflake notebooks are started inside their own staged directory
CANDIDATES += sorted(glob.glob("/mnt/**/pridictior_snow4", recursive=True))[:3]

ROOT = None
for c in CANDIDATES:
    if os.path.isdir(os.path.join(c, "src", "btcpred")):
        ROOT = os.path.abspath(c)
        break
if ROOT is None:                      # notebook staged on its own: src/ next to it
    ROOT = os.path.abspath("..") if os.path.isdir("../src") else os.path.abspath(".")

sys.path.insert(0, os.path.join(ROOT, "src"))
os.chdir(ROOT)
print("repo root :", ROOT)

from btcpred.env import detect, autoscale, describe
from btcpred.analyser import N_FEATURES
from btcpred.distributed import describe_plan

hw = detect()
auto = autoscale(hw, n_features=N_FEATURES)
print(describe(hw, auto))
print("plan      :", describe_plan(auto.world_size))

## 2 · Dataset (offline)The compact codec stores the whole second-by-second history in ~17 MB per month:prices as delta-encoded integer cents, OHLC and VWAP as offsets from the bar'sown close, taker flow as a ratio byte, and the timestamp implied by position ona continuous 1-second grid. Decoding needs nothing but NumPy.If the chunks live in an internal stage instead of next to the notebook, thesecond cell below pulls them down **once**; training then reads local disk only.

In [ ]:
from btcpred.compact import ChunkIndex

DATA_DIR = os.path.join(ROOT, "data", "btcusdt_1s")

# ---- optional: hydrate from an internal stage (run once per session) ------
STAGE = None            # e.g. "@BTCPRED.CORE.BTCPRED_DATA"
if STAGE and not os.path.exists(os.path.join(DATA_DIR, "MANIFEST.json")):
    from btcpred.snowflake_io import stage_to_local
    n = stage_to_local(STAGE, DATA_DIR)
    print(f"pulled {n} files from {STAGE}")

index = ChunkIndex(DATA_DIR)
print(index.describe())

# a single 30-minute window, exactly what one training epoch consumes
import datetime as dt
mid = (index.t_start + index.t_end) // 2
w = index.range(mid, mid + 1800)
print(f"sample window {dt.datetime.fromtimestamp(w.t0, dt.timezone.utc):%Y-%m-%d %H:%M} UTC"
      f" | {len(w)} bars | {w.close[0]:,.2f} -> {w.close[-1]:,.2f} USD"
      f" | silent seconds {w.gap.mean():.2%}")

## 3 · Build the trainerOne **epoch = 30 minutes of simulated market time = 1800 one-second bars**.The predictor emits a forecast **every second**. The label for a forecast made atsecond *t* only exists at *t+1800*, i.e. inside the *next* window — so window *k*is scored and back-propagated at the end of window *k+1*. All 1800 per-secondpredictions are stored first, then turned into the reward/loss signal for thenext cycle. A checkpoint is written every window.

In [ ]:
from btcpred.trainer import Trainer, TrainConfig
from btcpred.distributed import DistContext

cfg = TrainConfig(
    data_dir       = DATA_DIR,
    run_dir        = os.path.join(ROOT, "runs"),
    run_id         = time.strftime("snowflake_%Y%m%d_%H%M%S"),
    epoch_seconds  = 1800,      # 30 minutes of market per epoch
    horizon_seconds= 1800,      # predict 30 minutes ahead
    max_windows    = 120,       # raise this for a long run
    checkpoint_every = 1,       # every window, as specified
    lanes          = None,      # auto from hardware
    speed          = 0.0,       # 0 = max speed; set 600 to watch it crawl
    log_every      = 10,
)

ctx = DistContext(rank=0, world_size=1, local_rank=0, device=auto.device)
trainer = Trainer(cfg, ctx)
print(f"backend={trainer.model.backend}  params={trainer.model.n_params():,}  "
      f"tier={trainer.auto.tier}  lanes={trainer.n_lanes}  features={N_FEATURES}")
print("run dir:", trainer.store.root)

## 4 · Live training with a real-time graphThe chart redraws after every window:* **solid** — the actual market price as the simulator streams it;* **dotted** — the model's forecast, drawn at the time it is a prediction *for*  (shifted +30 min), so you read it as *"at this instant the model expected the  price over here to be X"*;* **shaded ribbon** — the model's own ±1σ uncertainty;* **progress bars** — accuracy, refreshed live.

In [ ]:
from btcpred.viz import LiveChart

chart = LiveChart(window_s=5400, notebook=True)
t0 = time.time()

for step in range(cfg.max_windows):
    row = trainer.step()
    if row is None:
        print("tape exhausted")
        break
    frame = trainer.latest_frame(lane=0)
    if frame and (step % 2 == 0 or step == cfg.max_windows - 1):
        chart.update(
            frame["ts"], frame["price"], frame["pred"], frame["sigma"],
            metrics=frame["metrics"],
            info=(f"{frame['utc']} UTC   window {frame['window']}   "
                  f"{frame['throughput']:,.0f} sim-s/s   "
                  f"loss {row.get('loss', float('nan')):.4f}"),
        )

print(f"\n{trainer.window_counter} windows "
      f"({trainer.sim_seconds/86400:.2f} simulated days) "
      f"in {time.time()-t0:.0f}s wall")

## 5 · Results: curves, held-out accuracy, artefacts

In [ ]:
from btcpred.metrics import render_bars
from btcpred.viz import plot_summary, build_standalone_html, animate_run
from IPython.display import HTML, Image, display

summary = trainer.finish()

print("training accuracy (EMA over the last windows)")
print(render_bars(trainer.running.values))
if summary.get("holdout"):
    print("\nheld-out slice (newest data, never trained on)")
    print(render_bars(summary["holdout"]))

run_dir = trainer.store.root
art = os.path.join(ROOT, "artifacts")
os.makedirs(art, exist_ok=True)

png  = plot_summary(run_dir, os.path.join(art, "training_curves.png"))
html = build_standalone_html(run_dir, os.path.join(art, "live_dashboard.html"))
display(Image(filename=png))
print("self-contained replay written to:", html)

In [ ]:
# Animated replay (GIF). Skip if you only want the interactive HTML above.
gif = animate_run(run_dir, os.path.join(art, "live_prediction.gif"),
                  frames=120, fps=12)
display(Image(filename=gif))

## 6 · Persist into Snowflake tables (optional)Writes the per-second prediction tape and per-window metrics into`BTC_PREDICTIONS` / `BTC_TRAIN_METRICS` so they can be queried with SQL orcharted in Snowsight. Skipped silently when no Snowpark session is active.

In [ ]:
try:
    from btcpred.snowflake_io import write_predictions, write_metrics
    n_pred = write_predictions(run_dir, "BTC_PREDICTIONS", max_windows=50)
    n_met  = write_metrics(run_dir, "BTC_TRAIN_METRICS")
    print(f"wrote {n_pred:,} prediction rows and {n_met} metric rows")
except Exception as e:
    print("skipped Snowflake write:", e)

## 7 · Scaling up| want | change ||---|---|| train on everything | `max_windows=None` (the tape is ~2.9 M windows) || watch it in real time | `speed=1` (real time) or `speed=600` (600× ) || bigger model | `max_tier="xxl"` — otherwise it is chosen from VRAM/RAM/cores || more GPU utilisation | `world_size=N` → N DDP ranks **sharing the same card** || resume | `TrainConfig(resume=True)` — reloads the newest checkpoint || longer horizon | `horizon_seconds=3600` (and `epoch_seconds` to match) |Checkpoints are in `runs/<run_id>/checkpoints/` (one per 30-minute window) andevery per-second prediction is in `runs/<run_id>/predictions/`.